In [2]:
%load_ext autoreload
%autoreload 2

## Imports e Configuração Inicial

In [3]:
import numpy as np
import pandas as pd
import re
import pickle
import math
from collections import Counter

# Importar as classes criadas no teu custom_nn.py
import sys
sys.path.append('../src')
from custom_nn import LogisticRegressionNumPy, DeepNeuralNetworkNumPy

# Definir seed para reprodutibilidade
np.random.seed(42)

## Funções de Pré-processamento e TF-IDF (100% em Python/NumPy)

In [4]:
def tokenize(text):
    """Limpa o texto e divide em palavras/tokens."""
    text = str(text).lower()
    tokens = re.findall(r'\b[a-z]{2,}\b', text) # Apenas palavras com 2+ letras
    return tokens

class SimpleTFIDF:
    def __init__(self, max_features=1000):
        self.max_features = max_features
        self.vocabulary_ = {}
        self.idf_ = {}

    def fit(self, raw_documents):
        # 1. Contar frequência de documentos para cada palavra
        doc_counts = Counter()
        total_docs = len(raw_documents)
        
        tokenized_docs = [tokenize(doc) for doc in raw_documents]
        for tokens in tokenized_docs:
            unique_tokens = set(tokens)
            for token in unique_tokens:
                doc_counts[token] += 1
                
        # 2. Selecionar as 'max_features' palavras mais frequentes
        most_common = doc_counts.most_common(self.max_features)
        self.vocabulary_ = {word: idx for idx, (word, _) in enumerate(most_common)}
        
        # 3. Calcular IDF: log(N / df)
        for word, idx in self.vocabulary_.items():
            df = doc_counts[word]
            self.idf_[word] = math.log((total_docs + 1) / (df + 1)) + 1

    def transform(self, raw_documents):
        X = np.zeros((len(raw_documents), len(self.vocabulary_)))
        for i, doc in enumerate(raw_documents):
            tokens = tokenize(doc)
            tf_counts = Counter(tokens)
            doc_len = len(tokens) if len(tokens) > 0 else 1
            
            for token, count in tf_counts.items():
                if token in self.vocabulary_:
                    idx = self.vocabulary_[token]
                    tf = count / doc_len
                    X[i, idx] = tf * self.idf_[token]
        return X

    def fit_transform(self, raw_documents):
        self.fit(raw_documents)
        return self.transform(raw_documents)

## Divisão dos Dados (Train / Val / Test) em NumPy

In [5]:
# Carregar o dataset unificado da Tarefa 1
df = pd.read_csv('../data/processed/full_dataset_updated.csv')
print(f"Total de amostras: {len(df)}")

# Reorganizar aleatoriamente (shuffle)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

texts = df['text'].values
labels = df['label'].values

# Separar em 80% Treino, 10% Validação, 10% Teste
n = len(df)
train_end = int(n * 0.8)
val_end = int(n * 0.9)

train_texts, train_labels = texts[:train_end], labels[:train_end]
val_texts, val_labels = texts[train_end:val_end], labels[train_end:val_end]
test_texts, test_labels = texts[val_end:], labels[val_end:]

print(f"Treino: {len(train_texts)} | Validação: {len(val_texts)} | Teste: {len(test_texts)}")

# Extrair Features TF-IDF (1000 features tabulares)
vectorizer = SimpleTFIDF(max_features=1000)
X_train = vectorizer.fit_transform(train_texts)
X_val = vectorizer.transform(val_texts)
X_test = vectorizer.transform(test_texts)

y_train = train_labels
y_val = val_labels
y_test = test_labels

print(f"Formato da matriz de treino (X_train): {X_train.shape}")

Total de amostras: 2000
Treino: 1600 | Validação: 200 | Teste: 200
Formato da matriz de treino (X_train): (1600, 672)


## Funções de Métricas de Avaliação em NumPy

In [7]:
def compute_metrics(y_true, y_pred_prob, threshold=0.5):
    y_pred = (y_pred_prob >= threshold).astype(int).flatten()
    y_true = y_true.flatten()
    
    tp = np.sum((y_pred == 1) & (y_true == 1))
    tn = np.sum((y_pred == 0) & (y_true == 0))
    fp = np.sum((y_pred == 1) & (y_true == 0))
    fn = np.sum((y_pred == 0) & (y_true == 1))
    
    accuracy = (tp + tn) / len(y_true) if len(y_true) > 0 else 0
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    
    return {
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1-Score': f1
    }

## Treino do Modelo Baseline (Regressão Logística em NumPy)

In [8]:
print("--- TREINANDO REGRESSÃO LOGÍSTICA (BASELINE) ---")

# 1. Dicionário de mapeamento
label_map = {'human': 0, 'chatgpt': 1, 'AI': 1, 'Human': 0}

# 2. Converte os rótulos de forma segura para NumPy (funciona para Pandas Series ou NumPy Array)
y_train_clean = np.array([label_map.get(val, val) for val in y_train], dtype=int)
y_val_clean = np.array([label_map.get(val, val) for val in y_val], dtype=int)

# 3. Garantir que X é NumPy
X_train_clean = np.asarray(X_train)
X_val_clean = np.asarray(X_val)

# 4. Inicializar e Treinar
log_reg = LogisticRegressionNumPy(input_dim=X_train_clean.shape[1], lr=0.1)
log_reg.fit(X_train_clean, y_train_clean, epochs=300)

# 5. Avaliação
val_preds_lr = log_reg.predict_proba(X_val_clean)
metrics_lr = compute_metrics(y_val_clean, val_preds_lr)

print("Resultados no Conjunto de Validação (Regressão Logística):")
for metric, val in metrics_lr.items():
    print(f"  {metric}: {val:.4f}")

--- TREINANDO REGRESSÃO LOGÍSTICA (BASELINE) ---
Resultados no Conjunto de Validação (Regressão Logística):
  Accuracy: 0.6850
  Precision: 0.0000
  Recall: 0.0000
  F1-Score: 0.0000


## Treino do Modelo Principal (Deep Neural Network em NumPy)

In [9]:
print("--- TREINANDO REDE NEURONAL PROFUNDA (DNN) ---")

# Garantia de segurança (caso não tenha corrido a célula anterior):
# Mapeamento e conversão de X e y para NumPy/Inteiros
label_map = {'human': 0, 'chatgpt': 1, 'AI': 1, 'Human': 0}

X_train_clean = np.asarray(X_train)
X_val_clean = np.asarray(X_val)

y_train_clean = np.array([label_map.get(val, val) for val in y_train], dtype=int)
y_val_clean = np.array([label_map.get(val, val) for val in y_val], dtype=int)

# Arquitetura: Input (1000) -> Oculta 1 (64) -> Oculta 2 (32) -> Output (1)
layer_dims = [X_train_clean.shape[1], 64, 32, 1]
dnn = DeepNeuralNetworkNumPy(layers_dims=layer_dims, dropout_rate=0.2)

epochs = 500
learning_rate = 0.05

for epoch in range(1, epochs + 1):
    # Passando os dados limpos (arrays NumPy)
    dnn.train_step(X_train_clean, y_train_clean, lr=learning_rate)
    
    if epoch % 50 == 0:
        val_preds_dnn, _ = dnn.forward(X_val_clean, training=False)
        val_metrics = compute_metrics(y_val_clean, val_preds_dnn)
        print(f"Epoch {epoch}/{epochs} - Val Acc: {val_metrics['Accuracy']:.4f} | Val F1: {val_metrics['F1-Score']:.4f}")

--- TREINANDO REDE NEURONAL PROFUNDA (DNN) ---
Epoch 50/500 - Val Acc: 0.6850 | Val F1: 0.0000
Epoch 100/500 - Val Acc: 0.6850 | Val F1: 0.0000
Epoch 150/500 - Val Acc: 0.6850 | Val F1: 0.0000
Epoch 200/500 - Val Acc: 0.6850 | Val F1: 0.0000
Epoch 250/500 - Val Acc: 0.6850 | Val F1: 0.0000
Epoch 300/500 - Val Acc: 0.7300 | Val F1: 0.2500
Epoch 350/500 - Val Acc: 0.9500 | Val F1: 0.9138
Epoch 400/500 - Val Acc: 1.0000 | Val F1: 1.0000
Epoch 450/500 - Val Acc: 1.0000 | Val F1: 1.0000
Epoch 500/500 - Val Acc: 1.0000 | Val F1: 1.0000


## Avaliação Final no Conjunto de Teste e Comparação

In [10]:
# 1. Tratamento e Mapeamento dos Dados de Teste
label_map = {'human': 0, 'chatgpt': 1, 'AI': 1, 'Human': 0}

X_test_clean = np.asarray(X_test)
y_test_clean = np.array([label_map.get(val, val) for val in y_test], dtype=int)

# 2. Testar a Regressão Logística
test_preds_lr = log_reg.predict_proba(X_test_clean)
test_metrics_lr = compute_metrics(y_test_clean, test_preds_lr)

# 3. Testar a DNN
test_preds_dnn, _ = dnn.forward(X_test_clean, training=False)
test_metrics_dnn = compute_metrics(y_test_clean, test_preds_dnn)

# 4. Apresentar Tabela Comparativa
results_df = pd.DataFrame(
    [test_metrics_lr, test_metrics_dnn], 
    index=['Logistic Regression', 'Custom DNN (NumPy)']
)

print("\n=== RESULTADOS FINAIS NO CONJUNTO DE TESTE ===")
print(results_df)


=== RESULTADOS FINAIS NO CONJUNTO DE TESTE ===
                     Accuracy  Precision  Recall  F1-Score
Logistic Regression      0.69        0.0     0.0       0.0
Custom DNN (NumPy)       1.00        1.0     1.0       1.0


## Guardar o Modelo e o Vetorizador para a Submissão

In [11]:
# Guardar o vetorizador TF-IDF e a DNN treinada
model_artifacts = {
    'vectorizer': vectorizer,
    'model_type': 'numpy_dnn',
    'params': dnn.params,
    'layer_dims': layer_dims
}

with open('../models/numpy_dnn.pkl', 'wb') as f:
    pickle.dump(model_artifacts, f)

print("Modelo NumPy e vetorizador gravados com sucesso em '../models/numpy_dnn.pkl'!")

Modelo NumPy e vetorizador gravados com sucesso em '../models/numpy_dnn.pkl'!
